# Notebook 03 — Model Training & Evaluation
**Crop Disease Detection System** · IILM University, Greater Noida · 2025–26

This notebook trains the Decision Tree classifier and generates all evaluation outputs.

**Contents:**
1. Load feature matrix
2. Train-test split (stratified 80/20)
3. [Optional] GridSearchCV hyperparameter tuning
4. Train final Decision Tree model
5. Evaluate: Accuracy, Precision, Recall, F1
6. Confusion matrix heatmap
7. Feature importance bar chart
8. Decision Tree visualization
9. Sample prediction with decision rules
10. Save model

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
sns.set_theme(style='darkgrid')
os.makedirs('../outputs', exist_ok=True)

RANDOM_STATE = 42
print('All imports successful.')

## 1 — Load Feature Matrix

In [ ]:
df = pd.read_csv('../data/features.csv')
print(f'Dataset shape: {df.shape}')
print(f'Class distribution:')
print(df['label'].value_counts())

FEATURE_NAMES = ['contrast', 'energy', 'homogeneity', 'correlation',
                 'red_mean', 'green_mean', 'blue_mean',
                 'red_std', 'green_std', 'blue_std']

X = df[FEATURE_NAMES].values.astype(np.float32)
y_raw = df['label'].values

le = LabelEncoder()
y = le.fit_transform(y_raw)
CLASS_NAMES = list(le.classes_)

print(f'\nFeature matrix X shape: {X.shape}')
print(f'Classes: {CLASS_NAMES}')

## 2 — Stratified Train-Test Split (80/20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f'Training samples : {len(X_train)}  ({len(X_train)/len(X)*100:.1f}%)')
print(f'Test samples     : {len(X_test)}   ({len(X_test)/len(X)*100:.1f}%)')

# Confirm stratification
from collections import Counter
train_dist = Counter([le.classes_[i] for i in y_train])
test_dist  = Counter([le.classes_[i] for i in y_test])
print(f'\nTrain class distribution: {dict(sorted(train_dist.items()))}')
print(f'Test class distribution:  {dict(sorted(test_dist.items()))}')

## 3 — [Optional] GridSearchCV Hyperparameter Tuning
Set `RUN_GRID_SEARCH = True` to run the full grid search (takes 5–15 minutes).
Default uses the optimal parameters found in the project synopsis.

In [ ]:
RUN_GRID_SEARCH = False   # Set to True to run full grid search

if RUN_GRID_SEARCH:
    param_grid = {
        'criterion':         ['gini', 'entropy'],
        'max_depth':         [5, 8, 10, 12, 15, 20],
        'min_samples_split': [2, 5, 10, 20],
        'min_samples_leaf':  [1, 3, 5, 10],
        'max_features':      ['sqrt', 'log2', None],
        'ccp_alpha':         [0.0, 0.001, 0.005],
    }
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    grid_search = GridSearchCV(
        DecisionTreeClassifier(random_state=RANDOM_STATE),
        param_grid=param_grid, cv=cv,
        scoring='accuracy', n_jobs=-1, verbose=1
    )
    grid_search.fit(X_train, y_train)
    best_params = grid_search.best_params_
    print(f'Best CV Accuracy: {grid_search.best_score_:.4f}')
    print(f'Best parameters: {best_params}')
else:
    # Optimal parameters from project synopsis
    best_params = {
        'criterion': 'gini', 'max_depth': 12,
        'min_samples_split': 10, 'min_samples_leaf': 5,
        'max_features': 'sqrt', 'ccp_alpha': 0.001
    }
    print('Using pre-determined optimal hyperparameters:')
    for k, v in best_params.items():
        print(f'  {k:<22} = {v}')

## 4 — Train Final Decision Tree Model

In [ ]:
clf = DecisionTreeClassifier(random_state=RANDOM_STATE, **best_params)
clf.fit(X_train, y_train)

print(f'Model trained successfully!')
print(f'  Tree depth       : {clf.get_depth()}')
print(f'  Number of leaves : {clf.get_n_leaves()}')
print(f'  Number of nodes  : {clf.tree_.node_count}')

# 5-fold cross-validation on training set
cv_scores = cross_val_score(clf, X_train, y_train, cv=5, scoring='accuracy')
print(f'\n5-fold CV accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'  Scores: {[round(s,4) for s in cv_scores]}')

## 5 — Evaluate: Accuracy, Precision, Recall, F1

In [ ]:
y_pred = clf.predict(X_test)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='weighted')
rec  = recall_score(y_test, y_pred, average='weighted')
f1   = f1_score(y_test, y_pred, average='weighted')

print('=' * 50)
print('  FINAL TEST SET RESULTS')
print('=' * 50)
print(f'  Accuracy  : {acc*100:.2f}%')
print(f'  Precision : {prec*100:.2f}%')
print(f'  Recall    : {rec*100:.2f}%')
print(f'  F1-Score  : {f1*100:.2f}%')
print('=' * 50)
print()
print('Per-Class Classification Report:')
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

## 6 — Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d',
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    cmap='YlOrRd',
    linewidths=0.5, linecolor='white',
    ax=ax
)
ax.set_title('Confusion Matrix — Decision Tree Classifier\nCrop Disease Detection System',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=11)
ax.set_ylabel('True Label', fontsize=11)
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('../outputs/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → outputs/confusion_matrix.png')

## 7 — Feature Importance Bar Chart

In [ ]:
importances = clf.feature_importances_
indices = np.argsort(importances)[::-1]
sorted_names = [FEATURE_NAMES[i] for i in indices]
sorted_vals  = importances[indices]

colors = ['#ef4444' if i < 2 else '#f59e0b' if i < 4 else '#3b82f6'
          for i in range(len(sorted_names))]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(sorted_names[::-1], sorted_vals[::-1], color=colors[::-1])

for bar, val in zip(bars, sorted_vals[::-1]):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

ax.set_xlabel('Feature Importance (Gini Gain Contribution)', fontsize=11)
ax.set_title('Feature Importance — Decision Tree\nCrop Disease Detection System',
             fontsize=12, fontweight='bold')
ax.axvline(x=0.1, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('../outputs/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Feature Rankings:')
for i, (name, val) in enumerate(zip(sorted_names, sorted_vals), 1):
    print(f'  {i:2}. {name:<20} {val:.4f}')

## 8 — Decision Tree Visualization
Rendering the top 4 levels of the tree for readability.

In [ ]:
fig, ax = plt.subplots(figsize=(22, 11))
plot_tree(
    clf,
    feature_names=FEATURE_NAMES,
    class_names=CLASS_NAMES,
    filled=True,
    rounded=True,
    fontsize=8,
    max_depth=4,        # top 4 levels only — full tree has more
    impurity=True,
    proportion=False,
    ax=ax
)
plt.title(
    'Decision Tree — Top 4 Levels (Full tree may be deeper)\n'
    'Crop Disease Detection System | DiaMOS Plant Dataset',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('../outputs/decision_tree_plot.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved → outputs/decision_tree_plot.png')

## 9 — Sample Prediction with Decision Rules
Demonstrate the interpretability advantage: traceable IF-THEN rules.

In [ ]:
from sklearn.tree import _tree

def trace_decision_path(clf, sample, feature_names, class_names):
    tree = clf.tree_
    node_ids = clf.decision_path(sample.reshape(1, -1)).indices
    print('Decision Path:')
    for i, node_id in enumerate(node_ids):
        if tree.children_left[node_id] == _tree.TREE_LEAF:
            counts = tree.value[node_id][0]
            pred   = class_names[np.argmax(counts)]
            conf   = counts.max() / counts.sum() * 100
            print(f'  {'  '*i}→ PREDICTION: {pred}  (confidence: {conf:.1f}%)')
        else:
            feat_idx  = tree.feature[node_id]
            threshold = tree.threshold[node_id]
            feat_name = feature_names[feat_idx]
            val       = sample[feat_idx]
            op = '<=' if val <= threshold else '>'
            print(f'  {'  '*i}IF {feat_name} {op} {threshold:.4f}  (value: {val:.4f})')

# Take first 3 test samples and show their decision paths
for i in range(min(3, len(X_test))):
    true_class = CLASS_NAMES[y_test[i]]
    pred_class = CLASS_NAMES[clf.predict(X_test[i].reshape(1, -1))[0]]
    print(f'\nSample {i+1}: True class = {true_class} | Predicted = {pred_class}')
    print('-' * 50)
    trace_decision_path(clf, X_test[i], FEATURE_NAMES, CLASS_NAMES)

## 10 — Save Model

In [ ]:
import pickle

os.makedirs('../models', exist_ok=True)
model_bundle = {'classifier': clf, 'label_encoder': le}

with open('../models/decision_tree_model.pkl', 'wb') as f:
    pickle.dump(model_bundle, f)

size_kb = os.path.getsize('../models/decision_tree_model.pkl') / 1024
print(f'Model saved → models/decision_tree_model.pkl  ({size_kb:.1f} KB)')
print()
print('SUMMARY:')
print(f'  Accuracy  : {acc*100:.2f}%')
print(f'  F1-Score  : {f1*100:.2f}%')
print(f'  Tree depth: {clf.get_depth()}')
print(f'  Leaves    : {clf.get_n_leaves()}')
print()
print('All outputs saved in outputs/ folder.')
print('Launch web app with: python app/app.py')